<a href="https://colab.research.google.com/github/GogBean/Lab/blob/main/exp4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [4]:
from scipy.sparse import data
#set random seed for reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")
#Load Dataset
data = load_breast_cancer()
X= pd.DataFrame(data.data, columns=data.feature_names)
y= data.target
print(f"Dataset Shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)} (0: Maligant, 1: Benign)")

#train-test split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

#standardize features
scaler= StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Preprocessing complete")


Dataset Shape: (569, 30)
Class distribution: [212 357] (0: Maligant, 1: Benign)
Preprocessing complete


In [7]:
mle_model =LogisticRegression(penalty=None,max_iter=10000)
mle_model.fit(X_train_scaled,y_train)
map_l2_model = LogisticRegression(penalty="l2",C=1.0,solver='lbfgs',max_iter=10000)
map_l2_model.fit(X_train_scaled,y_train)
map_l1_model=LogisticRegression(penalty='l1',solver='saga',C=1.0,max_iter=10000)
map_l1_model.fit(X_train_scaled,y_train)
weights_df = pd.DataFrame({
    'Feature': ['Intercept'] + list(data.feature_names),
    'MLE' : np.insert(mle_model.coef_[0],0,mle_model.intercept_[0]),
    'MAP_L2(Gaussian)' : np.insert(map_l2_model.coef_[0],0,map_l2_model.intercept_[0]),
    'MAP_L1(Laplace)' : np.insert(map_l1_model.coef_[0],0,map_l1_model.intercept_[0])
})
print(weights_df)


                    Feature         MLE  MAP_L2(Gaussian)  MAP_L1(Laplace)
0                 Intercept  -63.532600          0.445585         0.299321
1               mean radius    9.433096         -0.431904         0.000000
2              mean texture  -16.986388         -0.387326         0.000000
3            mean perimeter   40.020926         -0.393432         0.000000
4                 mean area   10.890150         -0.465210         0.000000
5           mean smoothness    5.033443         -0.071667         0.000000
6          mean compactness  274.940098          0.540164         0.000000
7            mean concavity -134.838158         -0.801458         0.000000
8       mean concave points -266.184736         -1.119804        -2.195998
9             mean symmetry   40.996505          0.236119         0.053600
10   mean fractal dimension -168.017946          0.075921         0.000000
11             radius error -259.639462         -1.268178        -2.447947
12            texture err

In [9]:
def evaluate(model,X,y,name):
    preds = model.predict(X)
    probs = model.predict_proba(X)[:,1]
    return {
        'Model': name,
        'Accuracy': accuracy_score(y,preds),
        'Precision': precision_score(y,preds),
        'Recall': recall_score(y,preds),
        'F1-Score': f1_score(y,preds),
        'AUC-ROC': roc_auc_score(y,probs)
    }
results = [
  evaluate(mle_model,X_test,y_test,'MLE(No Regularization)'),
  evaluate(map_l2_model,X_test,y_test,'MAP (L2 Regularization)'),
  evaluate(map_l1_model,X_test,y_test,'MAP (L1 Regularization)')
]
pd.DataFrame(results)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but LogisticRegression was fitted withou

,Model,Accuracy,Precision,Recall,F1-Score,AUC-ROC
0,MLE(No Regularization),0.377193,0.0,0.0,0.0,0.500000
1,MAP (L2 Regularization),0.377193,0.0,0.0,0.0,0.626761
2,MAP (L1 Regularization),0.377193,0.0,0.0,0.0,0.619718


In [11]:
plt.figure(figsize=(15,6))
melted_w=weights_df.melt(id_vars='Feature',value_vars=['MLE','MAP_L2(Gaussian)','MAP_L1(Laplace)'],
                         var_name='Method',value_name='Weight')
sns.barplot(data=melted_w[melted_w['Feature'] != 'Intercept'], x='Feature',y='weight',hue='method')
plt.xticks(rotation=90)
plt.title('Parameter Estimate Comparisons')
plt.tight_layout()
plt.show()

print("Number of zero weights in L1 (Laplace):", np.sum(map_l1_model.coef_[0]==0))

ValueError: Could not interpret value `weight` for `y`. An entry with this name does not appear in `data`.

<Figure size 1500x600 with 0 Axes>